# SISAL-OSM Mapper - Match Analysis + Map

Maps SISAL speleothem sites to OpenStreetMap cave data with **detailed matching statistics** and **interactive visualization**.

**Focus:** Find the **best OSM match** for each SISAL site (or none if nothing found)

**Author:** Florian Thiery  
**License:** MIT  
**Project:** EPICA-SISAL-FAIRification

## 1. Setup & Configuration

In [ ]:
# Import required libraries
import pandas as pd
import re
from typing import List, Tuple, Dict, Optional
import json
import requests
import time
from urllib.parse import urlencode
from pathlib import Path
import math
from difflib import SequenceMatcher
from IPython.display import HTML, display, IFrame
import folium
from folium.plugins import MarkerCluster

# Configuration
CSV_PATH = Path("sisal_sites_all.csv")  # Adjust if needed
OUTPUT_DIR = Path("output")
SEARCH_RADIUS = 5000  # meters (5 km)
API_DELAY = 1.5  # seconds between requests

# Create output directory
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ Setup complete")
print(f"  CSV Path: {CSV_PATH}")
print(f"  Output Directory: {OUTPUT_DIR}")
print(f"  Search Radius: {SEARCH_RADIUS/1000:.1f} km")

## 2. Utility Functions

In [ ]:
def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate distance between two coordinates in kilometers."""
    R = 6371  # Earth's radius in kilometers
    
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    delta_lat = math.radians(lat2 - lat1)
    delta_lon = math.radians(lon2 - lon1)
    
    a = (math.sin(delta_lat / 2) ** 2 + 
         math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(delta_lon / 2) ** 2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    return R * c

def string_similarity(s1: str, s2: str) -> float:
    """Calculate normalized string similarity (0.0 to 1.0)."""
    if not s1 or not s2:
        return 0.0
    
    # Normalize: lowercase and remove common suffixes
    s1_norm = s1.lower().replace(' cave', '').replace(' höhle', '').strip()
    s2_norm = s2.lower().replace(' cave', '').replace(' höhle', '').strip()
    
    return SequenceMatcher(None, s1_norm, s2_norm).ratio()

# Test functions
print("Test Haversine Distance:")
print(f"  Frankfurt <-> Mainz: {haversine_distance(50.1109, 8.6821, 50.0, 8.2715):.2f} km")

print("\nTest String Similarity:")
print(f"  'Bunker cave' vs 'Bunker': {string_similarity('Bunker cave', 'Bunker'):.3f}")
print(f"  'Hölloch' vs 'Helloch': {string_similarity('Hölloch', 'Helloch'):.3f}")

## 3. Load SISAL Data

In [ ]:
def load_sisal_data(csv_path: Path) -> pd.DataFrame:
    """Load SISAL sites CSV and extract coordinates from WKT."""
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")
    
    df = pd.read_csv(csv_path)
    
    def parse_wkt_point(wkt: str) -> Tuple[Optional[float], Optional[float]]:
        match = re.search(r'POINT\(([+-]?\d+\.?\d*)\s+([+-]?\d+\.?\d*)\)', str(wkt))
        if match:
            lon, lat = float(match.group(1)), float(match.group(2))
            return (lat, lon)
        return (None, None)
    
    df[['latitude', 'longitude']] = df['wkt'].apply(
        lambda x: pd.Series(parse_wkt_point(x))
    )
    
    return df

# Load data
df = load_sisal_data(CSV_PATH)
print(f"✓ Loaded {len(df)} SISAL cave sites\n")

# Display sample
df[['site_id', 'site_name', 'latitude', 'longitude', 'n_d18o_samples', 'n_d13c_samples']].head(10)

## 4. Overpass API Functions

**⚠️ WICHTIG:** Die Overpass-Abfrage sucht nach:
- `natural=cave_entrance` (primär)
- `tourism=attraction` + `attraction=cave`
- `natural=sinkhole` (potenzielle Höhlen)

In [ ]:
def build_overpass_query(lat: float, lon: float, radius: int = 5000) -> str:
    """Build Overpass QL query for cave features near coordinates."""
    query = f"""
[out:json][timeout:25];
(
  // Natural caves
  node["natural"="cave_entrance"](around:{radius},{lat},{lon});
  way["natural"="cave_entrance"](around:{radius},{lat},{lon});
  relation["natural"="cave_entrance"](around:{radius},{lat},{lon});
  
  // Tourism caves
  node["tourism"="attraction"]["attraction"="cave"](around:{radius},{lat},{lon});
  way["tourism"="attraction"]["attraction"="cave"](around:{radius},{lat},{lon});
  
  // Sinkholes
  node["natural"="sinkhole"](around:{radius},{lat},{lon});
  way["natural"="sinkhole"](around:{radius},{lat},{lon});
);
out body;
>;
out skel qt;
"""
    return query.strip()

def query_overpass(query: str) -> Dict:
    """Execute Overpass API query."""
    try:
        response = requests.post(
            "https://overpass-api.de/api/interpreter",
            data={'data': query},
            timeout=30
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"⚠️  API Error: {e}")
        return {"elements": []}

def generate_overpass_turbo_url(query: str) -> str:
    """Generate overpass-turbo.eu URL for manual verification."""
    base_url = "https://overpass-turbo.eu/"
    params = {'Q': query, 'R': ''}
    return base_url + '?' + urlencode(params)

# Test query generation
test_site = df.iloc[1]  # Bunker cave
test_query = build_overpass_query(test_site['latitude'], test_site['longitude'])
test_url = generate_overpass_turbo_url(test_query)

print(f"Example: {test_site['site_name']}")
print(f"Overpass Query Preview (first 200 chars):")
print(test_query[:200] + "...")
print("\nOverpass Turbo URL:")
display(HTML(f'<a href="{test_url}" target="_blank" style="color: #0066cc; font-weight: bold;">→ Open in Overpass Turbo</a>'))

## 5. Match Analysis Function

**Ziel:** Finde den **besten Match** pro SISAL-Site basierend auf:
- 70% Namen-Ähnlichkeit
- 30% Nähe (Distanz)

In [ ]:
def analyze_osm_match(sisal_row: pd.Series, osm_elements: List[Dict], 
                      search_radius: int) -> Dict:
    """
    Analyze OSM matches and identify the BEST match.
    
    Returns:
        Dictionary with best match details or None if no match found
    """
    sisal_lat = sisal_row['latitude']
    sisal_lon = sisal_row['longitude']
    sisal_name = sisal_row['site_name']
    
    analysis = {
        'site_id': sisal_row['site_id'],
        'site_name': sisal_name,
        'total_osm_elements': len(osm_elements),
        'cave_entrances': [],
        'sinkholes': [],
        'tourism_caves': [],
        'has_match': False,
        'best_match': None,
        'match_reason': 'No OSM elements found'
    }
    
    if not osm_elements:
        return analysis
    
    # Analyze each OSM element
    for element in osm_elements:
        tags = element.get('tags', {})
        
        # Get coordinates
        if element.get('type') == 'node':
            osm_lat = element.get('lat')
            osm_lon = element.get('lon')
        elif 'center' in element:
            osm_lat = element['center']['lat']
            osm_lon = element['center']['lon']
        else:
            continue
        
        # Calculate metrics
        distance_km = haversine_distance(sisal_lat, sisal_lon, osm_lat, osm_lon)
        osm_name = tags.get('name', 'Unnamed')
        name_sim = string_similarity(sisal_name, osm_name)
        
        match_info = {
            'osm_id': element.get('id'),
            'osm_type': element.get('type'),
            'osm_name': osm_name,
            'distance_km': round(distance_km, 3),
            'name_similarity': round(name_sim, 3),
            'coordinates': (osm_lat, osm_lon)
        }
        
        # Categorize
        if tags.get('natural') == 'cave_entrance':
            match_info['category'] = 'cave_entrance'
            analysis['cave_entrances'].append(match_info)
        elif tags.get('natural') == 'sinkhole':
            match_info['category'] = 'sinkhole'
            analysis['sinkholes'].append(match_info)
        elif tags.get('tourism') == 'attraction':
            match_info['category'] = 'tourism_cave'
            analysis['tourism_caves'].append(match_info)
    
    # Find BEST match (prioritize name similarity)
    all_matches = (analysis['cave_entrances'] + 
                   analysis['sinkholes'] + 
                   analysis['tourism_caves'])
    
    if all_matches:
        analysis['has_match'] = True
        
        # Calculate composite score (70% name, 30% proximity)
        for match in all_matches:
            proximity_score = max(0, 1 - match['distance_km'] / (search_radius / 1000))
            match['score'] = (match['name_similarity'] * 0.7 + proximity_score * 0.3)
        
        # Select BEST match
        best = max(all_matches, key=lambda x: x['score'])
        analysis['best_match'] = best
        
        # Classify match quality
        if best['name_similarity'] > 0.8:
            analysis['match_reason'] = f"Strong name match (sim: {best['name_similarity']:.2f})"
        elif best['name_similarity'] > 0.5:
            analysis['match_reason'] = f"Moderate name match (sim: {best['name_similarity']:.2f})"
        elif best['distance_km'] < 1.0:
            analysis['match_reason'] = f"Close proximity ({best['distance_km']:.2f} km)"
        else:
            analysis['match_reason'] = f"Within search radius ({best['distance_km']:.2f} km)"
    
    return analysis

print("✓ Match analysis function defined")
print("  Focus: Best match per SISAL site (or none)")

## 6. Process SISAL Sites

**⚠️ Adjust `MAX_SITES` for testing:**
- Start with 10 sites for testing
- Set to `None` to process all 305 sites (~8 minutes)

In [ ]:
# Configuration
MAX_SITES = 10  # Set to None to process all sites

results = []
sites_to_process = df if MAX_SITES is None else df.head(MAX_SITES)
total = len(sites_to_process)

print(f"{'='*70}")
print(f"Processing {total} SISAL sites (search radius: {SEARCH_RADIUS/1000:.1f} km)")
print(f"{'='*70}\n")

for idx, (_, row) in enumerate(sites_to_process.iterrows(), 1):
    if pd.notna(row['latitude']) and pd.notna(row['longitude']):
        print(f"[{idx}/{total}] {row['site_name']} (ID: {row['site_id']})")
        
        # Query OSM
        query = build_overpass_query(row['latitude'], row['longitude'], SEARCH_RADIUS)
        osm_data = query_overpass(query)
        
        # Analyze
        analysis = analyze_osm_match(row, osm_data.get('elements', []), SEARCH_RADIUS)
        
        # Report
        if analysis['has_match']:
            best = analysis['best_match']
            print(f"    ✓ BEST Match: {best['osm_name']} "
                  f"({best['distance_km']:.2f} km, sim: {best['name_similarity']:.2f}, "
                  f"score: {best['score']:.3f})")
        else:
            print(f"    ✗ No match: {analysis['match_reason']}")
        
        # Store results
        result = {
            'site_id': row['site_id'],
            'site_name': row['site_name'],
            'latitude': row['latitude'],
            'longitude': row['longitude'],
            'n_d18o_samples': row['n_d18o_samples'],
            'n_d13c_samples': row['n_d13c_samples'],
            'has_osm_match': analysis['has_match'],
            'total_osm_elements': analysis['total_osm_elements'],
            'cave_entrances_count': len(analysis['cave_entrances']),
            'sinkholes_count': len(analysis['sinkholes']),
            'tourism_caves_count': len(analysis['tourism_caves']),
            'match_reason': analysis['match_reason'],
            'overpass_url': generate_overpass_turbo_url(query)
        }
        
        if analysis['best_match']:
            best = analysis['best_match']
            result.update({
                'best_match_name': best['osm_name'],
                'best_match_distance_km': best['distance_km'],
                'best_match_name_similarity': best['name_similarity'],
                'best_match_category': best['category'],
                'best_match_osm_id': f"{best['osm_type']}/{best['osm_id']}",
                'best_match_score': round(best['score'], 3),
                'best_match_lat': best['coordinates'][0],
                'best_match_lon': best['coordinates'][1]
            })
        
        results.append(result)
        
        # Respect API rate limits
        if idx < total:
            time.sleep(API_DELAY)

# Create results DataFrame
results_df = pd.DataFrame(results)
print(f"\n{'='*70}")
print("✓ Processing complete")
print(f"{'='*70}")

## 7. Summary Statistics

In [ ]:
total = len(results_df)
matched = results_df['has_osm_match'].sum()
match_rate = (matched / total * 100) if total > 0 else 0

print(f"{'='*70}")
print("SISAL-OSM MATCH ANALYSIS SUMMARY")
print(f"{'='*70}\n")

print(f"Total sites analyzed: {total}")
print(f"Sites with BEST match: {matched} ({match_rate:.1f}%)")
print(f"Sites without match: {total - matched} ({100-match_rate:.1f}%)\n")

print("OSM Elements Found:")
print(f"  Cave entrances: {results_df['cave_entrances_count'].sum()}")
print(f"  Sinkholes: {results_df['sinkholes_count'].sum()}")
print(f"  Tourism caves: {results_df['tourism_caves_count'].sum()}")

if matched > 0:
    matched_df = results_df[results_df['has_osm_match']]
    
    print(f"\nBest Match Distance Statistics:")
    print(f"  Min: {matched_df['best_match_distance_km'].min():.3f} km")
    print(f"  Max: {matched_df['best_match_distance_km'].max():.3f} km")
    print(f"  Mean: {matched_df['best_match_distance_km'].mean():.3f} km")
    print(f"  Median: {matched_df['best_match_distance_km'].median():.3f} km")
    
    print(f"\nBest Match Name Similarity Statistics:")
    print(f"  Min: {matched_df['best_match_name_similarity'].min():.3f}")
    print(f"  Max: {matched_df['best_match_name_similarity'].max():.3f}")
    print(f"  Mean: {matched_df['best_match_name_similarity'].mean():.3f}")
    print(f"  Median: {matched_df['best_match_name_similarity'].median():.3f}")

## 8. View Detailed Results

In [ ]:
# Display all results with color coding
results_df.style.apply(
    lambda x: ['background-color: #e8f5e9' if v else 'background-color: #ffebee' 
               for v in x], 
    subset=['has_osm_match']
).format({
    'best_match_distance_km': '{:.2f}',
    'best_match_name_similarity': '{:.2f}',
    'best_match_score': '{:.3f}'
}, na_rep='-')

## 9. Export Results

In [ ]:
# Save main results
output_csv = OUTPUT_DIR / "sisal_osm_matches.csv"
results_df.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"✓ Results saved to: {output_csv}")

# Save top matches if any
if matched > 0:
    top_matches = results_df[results_df['has_osm_match']].sort_values(
        'best_match_score', 
        ascending=False
    ).head(20)
    
    top_csv = OUTPUT_DIR / "top_20_matches.csv"
    top_matches.to_csv(top_csv, index=False, encoding='utf-8-sig')
    print(f"✓ Top matches saved to: {top_csv}")

## 10. 🗺️ Interactive Map with OSM Base Layer

**Zeigt:**
- 🔵 **SISAL Sites** (alle)
- 🔴 **OSM Best Matches** (nur wenn gefunden)
- ➡️ **Verbindungslinien** zwischen SISAL und OSM-Match
- 🔗 **Overpass-Turbo-Links** in Popups

In [ ]:
def create_sisal_osm_map(results_df: pd.DataFrame) -> folium.Map:
    """Create interactive map with SISAL sites and their best OSM matches."""
    
    # Calculate map center
    center_lat = results_df['latitude'].mean()
    center_lon = results_df['longitude'].mean()
    
    # Create base map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=3,
        tiles='OpenStreetMap'
    )
    
    # Add alternative base layers
    folium.TileLayer('CartoDB positron', name='Light Mode').add_to(m)
    folium.TileLayer('CartoDB dark_matter', name='Dark Mode').add_to(m)
    
    # Layer 1: SISAL Sites (all)
    sisal_group = folium.FeatureGroup(name='SISAL Sites (all)', show=True)
    
    for idx, row in results_df.iterrows():
        if pd.notna(row['latitude']) and pd.notna(row['longitude']):
            
            # Color based on match status
            color = 'green' if row['has_osm_match'] else 'blue'
            
            popup_html = f"""
            <div style="font-family: Arial; font-size: 12px; min-width: 250px;">
                <b style="color: {color}; font-size: 14px;">{row['site_name']}</b><br>
                <hr style="margin: 5px 0;">
                <b>Site ID:</b> {row['site_id']}<br>
                <b>δ¹⁸O samples:</b> {row['n_d18o_samples']}<br>
                <b>δ¹³C samples:</b> {row['n_d13c_samples']}<br>
                <b>Coords:</b> {row['latitude']:.4f}, {row['longitude']:.4f}<br>
                <hr style="margin: 5px 0;">
                <b>OSM Match:</b> {'✓ Found' if row['has_osm_match'] else '✗ None'}<br>
            """
            
            if row['has_osm_match']:
                popup_html += f"""
                <b>Best Match:</b> {row.get('best_match_name', 'N/A')}<br>
                <b>Distance:</b> {row.get('best_match_distance_km', 0):.2f} km<br>
                <b>Name Similarity:</b> {row.get('best_match_name_similarity', 0):.2f}<br>
                <b>Score:</b> {row.get('best_match_score', 0):.3f}<br>
                <b>Reason:</b> {row['match_reason']}<br>
                """
            else:
                popup_html += f"<b>Reason:</b> {row['match_reason']}<br>"
            
            popup_html += f"""
                <hr style="margin: 5px 0;">
                <a href="{row['overpass_url']}" target="_blank" style="color: #0066cc; font-weight: bold;">
                    🔍 Check in Overpass Turbo →
                </a>
            </div>
            """
            
            # Add SISAL marker
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=8,
                popup=folium.Popup(popup_html, max_width=350),
                tooltip=f"{row['site_name']} {'✓' if row['has_osm_match'] else '✗'}",
                color=color,
                fill=True,
                fillColor=color,
                fillOpacity=0.6,
                weight=2
            ).add_to(sisal_group)
    
    sisal_group.add_to(m)
    
    # Layer 2: OSM Best Matches (only matched sites)
    osm_group = folium.FeatureGroup(name='OSM Best Matches', show=True)
    
    matched_sites = results_df[results_df['has_osm_match']]
    
    for idx, row in matched_sites.iterrows():
        if 'best_match_lat' in row and pd.notna(row['best_match_lat']):
            
            osm_popup_html = f"""
            <div style="font-family: Arial; font-size: 12px; min-width: 250px;">
                <b style="color: #cc0000; font-size: 14px;">OSM: {row['best_match_name']}</b><br>
                <hr style="margin: 5px 0;">
                <b>Category:</b> {row['best_match_category']}<br>
                <b>OSM ID:</b> {row['best_match_osm_id']}<br>
                <b>Coords:</b> {row['best_match_lat']:.4f}, {row['best_match_lon']:.4f}<br>
                <hr style="margin: 5px 0;">
                <b>Matched to SISAL:</b> {row['site_name']}<br>
                <b>Distance:</b> {row['best_match_distance_km']:.2f} km<br>
                <b>Name Similarity:</b> {row['best_match_name_similarity']:.2f}<br>
                <b>Match Score:</b> {row['best_match_score']:.3f}<br>
                <hr style="margin: 5px 0;">
                <a href="https://www.openstreetmap.org/{row['best_match_osm_id']}" 
                   target="_blank" style="color: #0066cc; font-weight: bold;">
                    🌐 View in OSM →
                </a>
            </div>
            """
            
            # Add OSM marker
            folium.CircleMarker(
                location=[row['best_match_lat'], row['best_match_lon']],
                radius=6,
                popup=folium.Popup(osm_popup_html, max_width=350),
                tooltip=f"OSM: {row['best_match_name']}",
                color='red',
                fill=True,
                fillColor='red',
                fillOpacity=0.8,
                weight=2
            ).add_to(osm_group)
            
            # Draw connection line between SISAL and OSM
            folium.PolyLine(
                locations=[
                    [row['latitude'], row['longitude']],
                    [row['best_match_lat'], row['best_match_lon']]
                ],
                color='orange',
                weight=2,
                opacity=0.6,
                popup=f"{row['site_name']} ↔ {row['best_match_name']}<br>Distance: {row['best_match_distance_km']:.2f} km"
            ).add_to(osm_group)
    
    osm_group.add_to(m)
    
    # Add layer control
    folium.LayerControl(collapsed=False).add_to(m)
    
    # Add title
    title_html = f'''
    <div style="position: fixed; 
                top: 10px; left: 60px; width: 400px; 
                background-color: white; border:2px solid grey; z-index:9999; 
                font-size:13px; padding: 10px; border-radius: 5px;">
        <b style="font-size: 16px;">SISAL-OSM Mapper</b><br>
        <span style="color: green;">●</span> SISAL sites with OSM match ({matched_sites.shape[0]})<br>
        <span style="color: blue;">●</span> SISAL sites without match ({len(results_df) - matched_sites.shape[0]})<br>
        <span style="color: red;">●</span> OSM cave entrances (best matches)<br>
        <span style="color: orange;">━</span> Connection lines (SISAL ↔ OSM)<br>
        <hr style="margin: 5px 0;">
        <small>Click markers for details | Layer control: top right</small>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    return m

# Create and display map
print("Creating interactive map...")
sisal_map = create_sisal_osm_map(results_df)

# Save map
map_path = OUTPUT_DIR / "sisal_osm_map.html"
sisal_map.save(str(map_path))
print(f"✓ Map saved to: {map_path}")
print("\nMap features:")
print("  - OpenStreetMap base layer")
print("  - SISAL sites (green = match, blue = no match)")
print("  - OSM best matches (red circles)")
print("  - Connection lines (orange)")
print("  - Overpass Turbo links in popups")

# Display in notebook
sisal_map

## 11. Overpass Turbo Links für manuelle Verifikation

**Wichtig:** Diese Links öffnen die Overpass-Query direkt in overpass-turbo.eu!

In [ ]:
print("🔍 Overpass Turbo Links (first 10 sites):\n")

for idx, row in results_df.head(10).iterrows():
    status = "✓" if row['has_osm_match'] else "✗"
    bg_color = '#e8f5e9' if row['has_osm_match'] else '#ffebee'
    
    display(HTML(f"""
        <div style="margin: 10px 0; padding: 12px; border: 1px solid #ccc; 
                    border-radius: 5px; background: {bg_color}; 
                    font-family: Arial; font-size: 13px;">
            <b style="font-size: 15px;">{status} {row['site_name']}</b> 
            <span style="color: gray;">(ID: {row['site_id']})</span><br>
            <b>Coords:</b> {row['latitude']:.4f}, {row['longitude']:.4f}<br>
            <b>Status:</b> {row['match_reason']}<br>
            {'<b>Best Match:</b> ' + str(row.get('best_match_name', '')) + '<br>' if row['has_osm_match'] else ''}
            <a href="{row['overpass_url']}" target="_blank" 
               style="color: #0066cc; font-weight: bold; text-decoration: none;">
                🔍 → Open in Overpass Turbo (verify manually)
            </a>
        </div>
    """))

print("\n💡 Tip: Klicke auf die Links, um die Overpass-Query in deinem Browser zu öffnen!")

## ✅ Zusammenfassung

**Was wurde gemacht:**
1. ✓ SISAL-Daten geladen (305 Höhlenstandorte)
2. ✓ Overpass-API-Queries generiert (5 km Radius)
3. ✓ **Besten Match** pro Site identifiziert (oder keinen)
4. ✓ Match-Qualität bewertet (Distanz + Namen-Ähnlichkeit)
5. ✓ CSV exportiert (`sisal_osm_matches.csv`)
6. ✓ **Interaktive Karte** erstellt mit OSM-Base-Layer
7. ✓ **Overpass-Turbo-Links** für manuelle Verifikation

**Nächste Schritte:**
- Karte im Browser öffnen (`output/sisal_osm_map.html`)
- CSV in Excel analysieren
- Overpass-Links nutzen für manuelle Checks
- Radius anpassen falls zu wenig Matches
- Für deRSE26-Poster: Match-Rate berechnen